In [2]:
import argparse
import pandas as pd
import numpy as np
from preprocess import preprocess
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_recall_curve, precision_score, recall_score, average_precision_score, roc_curve, auc, confusion_matrix, mean_squared_error,classification_report
import time

import matplotlib.pyplot as plt
from keras.utils import to_categorical

trainset = pd.read_csv('./data/NSL-KDD/KDDTrain+.txt', sep=",", header=None)
testset = pd.read_csv('./data/NSL-KDD/KDDTest+.txt', sep=",", header=None)

#下面部分是GAN训练并且生成数据
processor = preprocess()
print("数据预处理....")
df_train, df_test, train_Normal, train_R2L, train_U2R, train_Dos, train_Probe,test_Normal, test_R2L, test_U2R, test_Dos, test_Probe,train_Attack,test_Attack = processor.create_df(df_train=trainset, df_test=testset)
# normal_df, R2L_df, R2L_df_train, R2L_df_test = processor.create_df(df_train=trainset, df_test=testset)
print("已完成数据预处理")


Using TensorFlow backend.


数据预处理....
已完成数据预处理


In [3]:
dt={'Dos':pd.Series([len(train_Dos),len(test_Dos)],index=['Train','Test']),
   'Probe':pd.Series([len(train_Probe),len(test_Probe)],index=['Train','Test']),
   'R2L':pd.Series([len(train_R2L),len(test_R2L)],index=['Train','Test']),
   'U2R':pd.Series([len(train_U2R),len(test_U2R)],index=['Train','Test']),
   'Normal':pd.Series([len(train_Normal),len(test_Normal)],index=['Train','Test']),
   'Total_attack':pd.Series([len(train_Attack),len(test_Attack)],index=['Train','Test']),
   'Total':pd.Series([len(df_train),len(df_test)],index=['Train','Test'])}
type_df=pd.DataFrame(dt)
cols = ['Dos','Probe','R2L','U2R','Normal','Total_attack','Total']
type_df = type_df[cols]
display(type_df)


,Dos,Probe,R2L,U2R,Normal,Total_attack,Total
Train,11656,45927,995,52,67343,58630,125973
Test,2421,7460,2885,67,9711,12833,22544


In [4]:
#加入生成数据的R2L类别二分类
generated_data = pd.read_csv('./output2/fake_examples.csv', sep=",", header=None)
generated_data = processor.gererated_preprocess(generated_data)
generated_data.head(3)

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_srv_count,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate,attack_type
0,0.023381,0.337760,0.543059,0.632417,0.060709,0.061058,0.061155,0.060931,0.061090,0.061215,...,0.136977,0.095528,0.103247,0.044616,0.103006,0.999941,1.000000,0.039759,0.975339,1
1,0.019404,0.428382,0.607096,0.675547,0.066396,0.066913,0.066426,0.065931,0.065706,0.068302,...,0.067395,0.706444,0.082226,0.990614,0.121055,0.092070,0.211256,0.225578,0.367397,1
2,0.017988,0.353297,0.622369,0.625998,0.058701,0.059895,0.061084,0.056650,0.055695,0.087155,...,-0.278023,0.999893,0.311403,0.823468,-0.762270,1.000000,-0.859975,0.999911,0.994451,1


In [5]:


#train_add_BiR2L = train_Normal.append(generated_data)

#train_add_BiR2L = processor.merge_df(df_train, generated_data)
#train_add_BiR2L = np.concatenate(df_train, generated_data)

train_BiR2L = train_Normal.append(train_R2L)
train_add_BiR2L = train_BiR2L.append(generated_data)
X_train_add_R2L,y_train_add_R2L = processor.split_df(train_add_BiR2L)

test_BiR2L = test_Normal.append(test_R2L)
X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)

X_train_add_R2L.shape

(73338, 41)

In [6]:
#全局二分类
df_train2 = df_train.append(generated_data)
X_train_Bi,y_train_Bi = processor.split_df(df_train2)
y_train_Bi = y_train_Bi.replace(4,1).replace(3,1).replace(2,1)

X_test_Bi,y_test_Bi = processor.split_df(df_test)
y_test_Bi = y_test_Bi.replace(4,1).replace(3,1).replace(2,1)




In [7]:
#Dos类二分类
train_BiDos = train_Normal.append(train_Dos)
X_train_Dos,y_train_Dos = processor.split_df(train_BiDos)
y_train_Dos = y_train_Dos.replace(2,1)

test_BiDos = test_Normal.append(test_Dos)
X_test_Dos,y_test_Dos = processor.split_df(test_BiDos)
y_test_Dos = y_test_Dos.replace(2,1)

In [8]:
#R2L类二分类
#train_BiR2L = train_Normal.append(train_R2L)
X_train_R2L,y_train_R2L = processor.split_df(train_BiR2L)


# test_BiR2L = test_Normal.append(test_R2L)
# X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)
X_train_R2L.shape

(68338, 41)

In [9]:
#U2R类二分类
train_BiU2R = train_Normal.append(train_U2R)
X_train_U2R,y_train_U2R = processor.split_df(train_BiU2R)
y_train_U2R = y_train_U2R.replace(4,1)

test_BiU2R = test_Normal.append(test_U2R)
X_test_U2R,y_test_U2R = processor.split_df(test_BiU2R)
y_test_U2R = y_test_U2R.replace(4,1)

In [10]:
#Probe类二分类
train_BiProbe = train_Normal.append(train_Probe)
X_train_Probe,y_train_Probe = processor.split_df(train_BiProbe)
y_train_Probe = y_train_Probe.replace(3,1)

test_BiProbe = test_Normal.append(test_Probe)
X_test_Probe,y_test_Probe = processor.split_df(test_BiProbe)
y_test_Probe = y_test_Probe.replace(3,1)

In [11]:
# X = X_train_add_R2L
# Y = y_train_add_R2L
# C = y_test_R2L
# T = X_test_R2L

X = X_train_Bi
Y = y_train_Bi
C = y_test_Bi
T = X_test_Bi

In [12]:
Y = Y.astype(int)


In [13]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)


# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]


scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)


model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")


D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


(22544,)
(22544,)
***************************************************************


In [14]:
model = LogisticRegression()
model.fit(traindata, trainlabel)
print(model)

# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)


cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)
(22544,)
(22544,)
population: 22544
P: 12833
N: 9711
PositiveTest: 8677
NegativeTest: 13867
TP: 8030
TN: 9064
FP: 647
FN: 4803
TPR: 0.625730538456
TNR: 0.933374523736
PPV: 0.9254350582
NPV: 0.653638133699
FPR: 0.066625476264
FDR: 0.0745649418002
FNR: 0.374269461544
ACC: 0.758250532292
F1_score: 0.746629474663
MCC: 0.569001540393
informedness: 0.559105062192
markedness: 0.579073191899
prevalence: 0.569242370476
LRP: 9.39176083299
LRN: 0.400985297998
DOR: 23.4217086758
FOR: 0.346361866301
Predicted  False  True  __all__
Actual                         
False       9064   647     9711
True        4803  8030    12833
__all__    13867  8677    22544
(22544,)
(22544,)
***************************************************************


In [15]:
# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

#expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


# cm = metrics.confusion_matrix(expected, predicted)
# print(cm)
# tpr = float(cm[0][0])/np.sum(cm[0])
# fpr = float(cm[1][1])/np.sum(cm[1])
# print("%.3f" %tpr)
# print("%.3f" %fpr)
# print("Accuracy")
# print("%.3f" %ACC)
# print("precision")
# print("%.3f" %precision)
# print("recall")
# print("%.3f" %recall)
# print("f-score")
# print("%.3f" %f1)
# print("fpr")
# print("%.3f" %fpr)
# print("tpr")
# print("%.3f" %tpr)
print("***************************************************************")



model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")

print("AdaBoostClassifier")
model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("end****end***********************************************************")




GaussianNB(priors=None, var_smoothing=1e-09)
(22544,)
(22544,)
population: 22544
P: 12833
N: 9711
PositiveTest: 12686
NegativeTest: 9858
TP: 8092
TN: 5117
FP: 4594
FN: 4741
TPR: 0.630561832775
TNR: 0.526928225723
PPV: 0.637868516475
NPV: 0.519070805437
FPR: 0.473071774277
FDR: 0.362131483525
FNR: 0.369438167225
ACC: 0.585920865862
F1_score: 0.634194129864
MCC: 0.157214449045
informedness: 0.157490058498
markedness: 0.156939321912
prevalence: 0.569242370476
LRP: 1.33290943798
LRN: 0.701116678117
DOR: 1.90112356414
FOR: 0.480929194563
Predicted  False   True  __all__
Actual                          
False       5117   4594     9711
True        4741   8092    12833
__all__     9858  12686    22544
(22544,)
(22544,)
***************************************************************
KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(22544,)
(22544,)
population: 22544
P: 12833
N:

In [16]:
model = svm.SVC(kernel='linear')#调参
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")

(22544,)
(22544,)
population: 22544
P: 12833
N: 9711
PositiveTest: 7878
NegativeTest: 14666
TP: 7612
TN: 9445
FP: 266
FN: 5221
TPR: 0.593158263851
TNR: 0.972608382247
PPV: 0.966235085047
NPV: 0.644006545752
FPR: 0.0273916177531
FDR: 0.033764914953
FNR: 0.406841736149
ACC: 0.756609297374
F1_score: 0.735068321182
MCC: 0.587583492592
informedness: 0.565766646098
markedness: 0.610241630799
prevalence: 0.569242370476
LRP: 21.6547364671
LRN: 0.418299639994
DOR: 51.768479809
FOR: 0.355993454248
Predicted  False  True  __all__
Actual                         
False       9445   266     9711
True        5221  7612    12833
__all__    14666  7878    22544
(22544,)
(22544,)
***************************************************************


In [17]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=41,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [18]:
#DNN
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
#csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
model = KerasClassifier(build_fn=build_model, epochs=100, batch_size=84)
#model.fit(traindata, trainlabel, callbacks=[checkpointer,csv_logger])
model.fit(traindata, trainlabel, callbacks=[checkpointer])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
print(predicted.shape)
print(expected.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/100
130973/130973 [==============================] - 6s 50us/step - loss: 0.0898 - acc: 0.9690

Epoch 00001: loss improved from inf to 0.08976, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/100
130973/130973 [==============================] - 5s 37us/step - loss: 0.0463 - acc: 0.9832

Epoch 00002: loss improved from 0.08976 to 0.04630, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/100
130973/130973 [==============================] - 5s 42us/step - loss: 0.0346 - acc: 0.9882

Epoch 00003: loss improved from 0.04630 to 0.03463, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/100
130973/130973 [==============================] - 5s 39us/step - loss: 0.0279 - acc: 0.9902

Epoch 00004: loss improved from 0.03463 to 0.02789, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/100
130973/130973 [==============================] - 5s 40us/step - loss: 0.0238 - acc: 0.9919

Epoch 00005: loss improved from 0.02789 to 0.02378, saving model to ./DNNResult/checkpoi

130973/130973 [==============================] - 5s 39us/step - loss: 0.0080 - acc: 0.9971

Epoch 00042: loss improved from 0.00806 to 0.00797, saving model to ./DNNResult/checkpoint-42.hdf5
Epoch 43/100
130973/130973 [==============================] - 5s 39us/step - loss: 0.0078 - acc: 0.9971

Epoch 00043: loss improved from 0.00797 to 0.00778, saving model to ./DNNResult/checkpoint-43.hdf5
Epoch 44/100
130973/130973 [==============================] - 5s 39us/step - loss: 0.0078 - acc: 0.9971

Epoch 00044: loss improved from 0.00778 to 0.00778, saving model to ./DNNResult/checkpoint-44.hdf5
Epoch 45/100
130973/130973 [==============================] - 5s 39us/step - loss: 0.0079 - acc: 0.9970

Epoch 00045: loss did not improve from 0.00778
Epoch 46/100
130973/130973 [==============================] - 5s 39us/step - loss: 0.0078 - acc: 0.9972

Epoch 00046: loss improved from 0.00778 to 0.00776, saving model to ./DNNResult/checkpoint-46.hdf5
Epoch 47/100
130973/130973 [=================

130973/130973 [==============================] - 6s 44us/step - loss: 0.0060 - acc: 0.9977

Epoch 00089: loss improved from 0.00611 to 0.00596, saving model to ./DNNResult/checkpoint-89.hdf5
Epoch 90/100
130973/130973 [==============================] - 5s 41us/step - loss: 0.0058 - acc: 0.9978

Epoch 00090: loss improved from 0.00596 to 0.00581, saving model to ./DNNResult/checkpoint-90.hdf5
Epoch 91/100
130973/130973 [==============================] - 5s 41us/step - loss: 0.0061 - acc: 0.9976

Epoch 00091: loss did not improve from 0.00581
Epoch 92/100
130973/130973 [==============================] - 5s 40us/step - loss: 0.0058 - acc: 0.9978

Epoch 00092: loss improved from 0.00581 to 0.00580, saving model to ./DNNResult/checkpoint-92.hdf5
Epoch 93/100
130973/130973 [==============================] - 5s 40us/step - loss: 0.0061 - acc: 0.9977

Epoch 00093: loss did not improve from 0.00580
Epoch 94/100
130973/130973 [==============================] - 5s 40us/step - loss: 0.0057 - acc: 